# KingFisher Presto

```{device-card} thermo-fisher-kingfisher-presto
```

| Property | Value |
|---|---|
| Manufacturer | Thermo Fisher Scientific |
| Instrument | KingFisher Presto magnetic particle processor |
| Connection | USB HID (`pylabrobot[hid]`) |
| API | `KingFisherPresto` |
| Units | Seconds, degrees Celsius, microliters |

```{warning}
This driver and the BDZ writer have not been verified on hardware or against manufacturer-generated BDZ fixtures. The interface specification and original examples referenced during development are unavailable. Report tested firmware and operations before removing this warning.
```

The instrument executes stored protocols and reports events. PLR handles connection, file transfer, explicit nest positioning, and operator acknowledgements. A successful start command means the instrument accepted the command; a terminal event reports the outcome.

See the manufacturer's [Momentum integration guide](https://apps.thermofisher.com/apps/lahr/LA_Online_Help_Resource/en-us/Content/Topics/Drivers/Extraction-Purification/DRVR%20KingFisher%20Presto.htm) for integration context. It does not replace the missing wire-protocol specification.

## Physical setup

Connect the Presto to its power supply and the computer by USB. Install the correct magnetic head, heater block, plates and tip comb for your protocol according to the instrument manual. Keep the turntable clear before moving it. Install the `hid` optional dependency and the operating system's HID library; on Linux, grant your user access to the USB HID device.

Nests are numbered 1 and 2. A nest can face the loading station or the processing head. PLR initially treats both positions as unknown.

## Connect

Import the device directly. Supply `serial_number="..."` if more than one Presto is connected. `setup()` reads identity and status without moving the turntable or clearing an instrument fault.

In [ ]:
from pylabrobot.thermo_fisher.kingfisher.presto import KingFisherPresto

presto = KingFisherPresto(timeout=30)
await presto.setup()
presto.instrument, presto.version, presto.serial

## Inspect status

`status` is `Idle`, `Busy`, or `In error`. An existing run remains under instrument control after a connection is opened.

In [ ]:
status = await presto.get_status()
status

## Move a nest to loading

This moves nest 1 to the loading station. The call waits for rotation completion. Repeating the same target does not swap nests.

In [ ]:
await presto.rotate(position=1, location="loading")
presto.get_turntable_state()

## Place the plate, then move it to processing

Place the protocol's plate in nest 1 before running the next cell. `load_plate(position=1)` means move that specific nest to processing. It does not detect or assign labware in a PLR resource tree.

In [ ]:
await presto.load_plate(position=1)

## List stored protocols

Inspect the names and memory utilization. Choose a protocol appropriate for the installed consumables before starting a run.

In [ ]:
names, memory_used = await presto.list_protocols()
names, memory_used

## Inspect protocol steps

Replace the example name with an existing stored protocol. Durations are returned in seconds; the returned tip and step names can be used for single-step execution.

In [ ]:
protocol_name = "Instrument Test Protocol"
duration = await presto.get_protocol_duration(protocol_name)
duration

## Download a protocol

Downloads return the original bytes so unsupported BindIt features are preserved. The structured `read_bdz()` codec covers a subset of Mix protocols and rejects unsupported step types; keep original files for lossless transfer.

In [ ]:
from pathlib import Path

original_bdz = await presto.download_protocol(protocol_name)
Path("presto-protocol-backup.bdz").write_bytes(original_bdz)

## Build a Mix protocol

The shared codec lives under `kingfisher.bdz`. This creates a protocol in memory; it does not move hardware. Review durations, temperatures, plate definitions, and tip-comb settings before using generated files. The codec has synthetic regression tests but no manufacturer fixtures.

In [ ]:
from pylabrobot.thermo_fisher.kingfisher.bdz import (
    BdzProtocol, MixStep, ProtocolTip, read_bdz, write_bdz,
)

example = BdzProtocol(
    name="plr-example-mix",
    tips=[ProtocolTip(name="Tip1", steps=[MixStep(name="Mix", shake_duration=30)])],
)
example_bdz = write_bdz(example)
read_bdz(example_bdz)

## Upload a protocol

Uploading overwrites a stored protocol with the same name. This accepts original BDZ bytes or a `BdzProtocol`; it rejects Duo files before sending them to the Presto. This cell uploads without starting execution.

In [ ]:
await presto.upload_protocol("plr-example-mix", example_bdz)

## Start a protocol

Ensure the instrument and consumables match the selected protocol. This starts a full run. It returns when the command is accepted; consume events to follow progress and handle plate changes.

In [ ]:
await presto.start_protocol(protocol_name)

## Read the next event

Run this cell repeatedly to inspect events. Progress events remain visible. `Ready`, `Aborted`, and `Error` are terminal; inspect Error details before clearing a fault. When reconnecting to an existing run, use `next_event(attach=True)` once to check current status first.

In [ ]:
event = await presto.next_event()
event.name, event.requires_acknowledgement, event.details.attrib

## Complete the requested action, then acknowledge

For `LoadPlate`, `RemovePlate`, `ChangePlate`, or `Pause`, perform the requested physical action before running this cell. Never automatically acknowledge these events in a loop. Faults use the separate `error_acknowledge()` operation.

In [ ]:
if event.requires_acknowledgement:
    await presto.acknowledge()

## Inspect remaining time

Run this while a protocol is active. Both values use seconds; a missing time-to-pause is `None`.

In [ ]:
await presto.get_protocol_time_left()

## Request execution to stop

This requests Stop and leaves the connection open. Continue reading events to observe the terminal outcome.

In [ ]:
await presto.stop_protocol()

## Execute a single named step

Use a tip and step returned by `get_protocol_duration()`. Run only when the instrument is ready and the required plate and tip comb are in place. Both arguments are required for a single step; do not run this cell during another run.

In [ ]:
step = duration.steps[0]
await presto.start_protocol(protocol_name, tip=step.tip, step=step.name)

## Acknowledge a fault

Inspect `get_status()` and resolve the reported condition at the instrument before explicitly acknowledging the fault.

In [ ]:
await presto.error_acknowledge()

## Abort

Use only when aborting an active operation is intended. Abort uses a HID feature report followed by the abort payload. The driver invalidates the connection afterwards so late responses cannot complete a later command. Close and reopen the connection before further operations; inspect status and physical state before resuming.

In [ ]:
await presto.abort()

## Reconnect after abort or a command timeout

A timeout or cancellation has an uncertain instrument outcome; it does not imply the physical operation stopped. Reconnecting clears local response/event state and reads instrument status without moving hardware.

In [ ]:
await presto.stop()
await presto.setup()
await presto.get_status()

## Remove a stored protocol

Delete the example only after its execution has finished and it is no longer needed.

In [ ]:
await presto.remove_protocol("plr-example-mix")

## Disconnect

`stop()` closes the connection. It does not stop an instrument protocol; use `stop_protocol()` when that is intended. The device also supports `async with KingFisherPresto() as presto:` for automatic connection cleanup.

In [ ]:
await presto.stop()